# 02 — Hedonic Pricing Model

## What Is a Hedonic Pricing Model?

A hedonic model treats a home's price as the **sum of the value of its individual characteristics**:

```
Price = f(sqft, beds, baths, lot_size, year_built, location, ...)
```

Once trained on real sold listings, the model can predict what *any* home **should** cost.
Homes listed below their predicted price are potentially undervalued.

## This Notebook:
1. Trains three candidate models (Linear Regression, Random Forest, XGBoost)
2. Evaluates them using hold-out test data
3. Interprets which features drive the model's predictions
4. Saves the best model for use in the daily pipeline

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

from src.scraper import load_all_raw_csv
from src.features import prepare_dataset, get_model_feature_cols
from src.model import train, load_model, predict, _build_candidates

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 1. Load and Prepare Data

In [ ]:
raw = load_all_raw_csv('../data/raw')
df = prepare_dataset(raw)

# Use sold listings as training data (they have confirmed final prices)
if 'sold' in df.columns:
    train_df = df[df['sold'] == True].copy()
    print(f'Training on sold listings: {len(train_df):,}')
    if len(train_df) < 50:
        print('  ⚠ Very few sold listings — using all listings instead.')
        train_df = df.copy()
else:
    train_df = df.copy()
    print(f'Training on all listings: {len(train_df):,}')

feature_cols = get_model_feature_cols(train_df)
print(f'Features: {len(feature_cols)}')
print(feature_cols)

## 2. Train/Test Split

In [ ]:
X = train_df[feature_cols].values
y = train_df['price'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples: {len(X_train):,}')
print(f'Test samples:     {len(X_test):,}')
print(f'Price range:      ${y.min():,.0f} — ${y.max():,.0f}')
print(f'Median price:     ${np.median(y):,.0f}')

## 3. Train & Compare All Candidate Models

In [ ]:
candidates = _build_candidates()
results = {}
fitted_models = {}

for name, pipeline in candidates.items():
    print(f'Training {name}...')
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE%': mape}
    fitted_models[name] = pipeline
    print(f'  RMSE=${rmse:,.0f}  MAE=${mae:,.0f}  R²={r2:.3f}  MAPE={mape:.1f}%\n')

results_df = pd.DataFrame(results).T.sort_values('RMSE')
print('\n--- MODEL COMPARISON ---')
print(results_df)

## 4. Metrics Visualized

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['RMSE', 'MAE', 'R2']
titles  = ['RMSE ($) — Lower is Better', 'MAE ($) — Lower is Better', 'R² — Higher is Better']
colors  = ['steelblue', 'seagreen', 'mediumpurple']

for ax, metric, title, color in zip(axes, metrics, titles, colors):
    vals = results_df[metric]
    bars = ax.bar(vals.index, vals.values, color=color, edgecolor='white')
    ax.set_title(title, fontsize=11)
    ax.set_ylabel(metric)
    for bar, val in zip(bars, vals.values):
        label = f'${val:,.0f}' if metric in ['RMSE', 'MAE'] else f'{val:.3f}'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                label, ha='center', va='bottom', fontsize=9)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Predicted vs Actual — Visual Accuracy Check

In [ ]:
best_name = results_df.index[0]  # lowest RMSE
best_model = fitted_models[best_name]
y_pred_best = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter: predicted vs actual
ax = axes[0]
ax.scatter(y_test / 1e3, y_pred_best / 1e3, alpha=0.5, s=25, color='steelblue')
lims = [min(y_test.min(), y_pred_best.min()) / 1e3 * 0.95,
        max(y_test.max(), y_pred_best.max()) / 1e3 * 1.05]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Price ($K)')
ax.set_ylabel('Predicted Price ($K)')
ax.set_title(f'{best_name}: Predicted vs Actual')
ax.legend()

# Residuals
ax = axes[1]
residuals = (y_pred_best - y_test) / 1e3
ax.hist(residuals, bins=40, color='salmon', edgecolor='white')
ax.axvline(0, color='black', linewidth=1.5)
ax.axvline(residuals.mean(), color='red', linestyle='--',
           label=f'Mean error: ${residuals.mean():.1f}K')
ax.set_xlabel('Prediction Error ($K)')
ax.set_ylabel('Count')
ax.set_title('Residual Distribution')
ax.legend()

plt.tight_layout()
plt.show()

print(f'Best model: {best_name}')
print(f'Typical error: ±${np.abs(residuals).median():.1f}K')

## 6. Feature Importance — What Drives Prices?

In [ ]:
# Use permutation importance — works for any model type
print('Computing permutation importance (this takes ~30s)...')
perm = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=10, random_state=42, n_jobs=-1
)

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': perm.importances_mean,
    'std': perm.importances_std,
}).sort_values('importance', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(importance_df['feature'], importance_df['importance'],
        xerr=importance_df['std'], color='mediumpurple', edgecolor='white', capsize=3)
ax.set_xlabel('Mean Decrease in R² when feature is shuffled')
ax.set_title(f'{best_name} — Top Feature Importances\n(Larger = More Important)')
plt.tight_layout()
plt.show()

## 7. Learning Curve — Do We Have Enough Data?

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    best_model, X, y, cv=5,
    train_sizes=np.linspace(0.1, 1.0, 8),
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_sizes, -train_scores.mean(axis=1) / 1e3, 'o-', color='steelblue', label='Training RMSE')
ax.fill_between(train_sizes,
                (-train_scores.mean(axis=1) - train_scores.std(axis=1)) / 1e3,
                (-train_scores.mean(axis=1) + train_scores.std(axis=1)) / 1e3,
                alpha=0.2, color='steelblue')
ax.plot(train_sizes, -val_scores.mean(axis=1) / 1e3, 'o-', color='coral', label='Validation RMSE')
ax.fill_between(train_sizes,
                (-val_scores.mean(axis=1) - val_scores.std(axis=1)) / 1e3,
                (-val_scores.mean(axis=1) + val_scores.std(axis=1)) / 1e3,
                alpha=0.2, color='coral')
ax.set_xlabel('Training Examples')
ax.set_ylabel('RMSE ($K)')
ax.set_title('Learning Curve — Does More Data Help?')
ax.legend()
plt.tight_layout()
plt.show()

print("If validation RMSE is still decreasing at the right edge,")
print("getting more listings data would improve the model.")

## 8. Save the Best Model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

model_path   = '../models/hedonic_model.joblib'
feature_path = '../models/hedonic_model_features.joblib'

joblib.dump(best_model, model_path)
joblib.dump(feature_cols, feature_path)

print(f'Model saved to {model_path}')
print(f'Features saved to {feature_path}')
print(f'Best model: {best_name}')
print(f'RMSE on test set: ${results_df.loc[best_name, "RMSE"]:,.0f}')